# Phase 2: Thai Equity Data Pipeline
ดึงราคาหุ้นไทยจาก yfinance, ตรวจความสะอาดของข้อมูล, แล้วบันทึกลง SQLite (`data/prices.db`)

In [1]:
import sys
import yfinance as yf
import pandas as pd

print("Python path:", sys.executable)

# ทดสอบดึงข้อมูลหุ้น PTT
df = yf.download("PTT.BK", start="2020-01-01", end="2024-01-01")
df.head()

Python path: /Users/tim/gemini_vscode/Carreer_path/CV_round2/ttb_T00013266/Thai_Equity_Portfolio/.venv/bin/python


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PTT.BK,PTT.BK,PTT.BK,PTT.BK,PTT.BK
Date,,,,,
2020-01-02,30.823532,30.823532,29.807372,29.976732,59084700
2020-01-03,31.500978,31.839699,31.162258,31.331618,147370600
2020-01-06,31.839691,32.347771,31.500971,32.009051,91026800
2020-01-07,32.009056,32.009056,31.331616,32.009056,60059600
2020-01-08,31.670332,32.347772,31.670332,32.178412,57759900


In [3]:
# 1. ลองดึงหุ้น 16 ตัว + SET Index ย้อนหลัง 10 ปี (2015-2024)
tickers = [
    "PTT.BK", "PTTEP.BK", "KBANK.BK", "SCB.BK", "BBL.BK", "KTB.BK",
    "CPALL.BK", "BJC.BK", "ADVANC.BK", "AOT.BK", "MINT.BK", "BDMS.BK",
    "BH.BK", "SCC.BK", "CPN.BK", "DELTA.BK", "TDEX.BK"
]

raw_data = yf.download(tickers, start="2015-01-01", end="2024-12-31", auto_adjust=False)

# 2. ดึงเฉพาะราคา Adjusted Close ออกมาเช็ก missing
adj_close = raw_data['Adj Close']
missing_counts = adj_close.isna().sum()
print("จำนวนวันที่ข้อมูลขาดหายไป (Missing Days):")
print(missing_counts[missing_counts > 0])

[*********************100%***********************]  17 of 17 completed

จำนวนวันที่ข้อมูลขาดหายไป (Missing Days):
Ticker
SCB.BK     1778
TDEX.BK       1
dtype: int64


### Notes: ปัญหาที่เจอในข้อมูล & วิธีแก้
- `^SET.BK`: Yahoo ไม่มีประวัติราคาย้อนหลัง 10 ปี โหลดแล้วว่างเปล่า -> ใช้ `TDEX.BK` (SET50 ETF) เป็น benchmark แทน ได้ทั้งประวัติครบและสะท้อน Total Return จริงของตลาด
- `SCB.BK`: หายไป 1,778 วัน เพราะปี 2022 SCB ปรับโครงสร้างเป็น SCBX ใน Yahoo เลยมีประวัติแค่ตั้งแต่ เม.ย. 2022 เป็นต้นมา -> เปลี่ยนไปใช้ `TTB.BK` แทน ข้อมูลครบ 10 ปีเต็ม (2,430 วัน)
- `TDEX.BK`: ขาดไป 1 วันเพราะวันหยุดไม่ตรงกัน -> ใช้ `ffill()` ลอกราคาปิดวันก่อนหน้ามาแทน

### Notes: ทำความสะอาดข้อมูล & ปรับทรงตารางเตรียมลง SQL
- `ffill().bfill()`:
  - `ffill`: วันไหนไม่มีราคา (วันหยุดตลาด/ไม่มีเทรด) ให้ใช้ราคาปิดวันก่อนหน้า (ตามหลักการเงิน วันที่ไม่ได้เทรด มูลค่าพอร์ต = วันก่อน)
  - `bfill`: กันเหนียวเผื่อวันแรกสุดไม่มีราคา ให้ดึงวันถัดไปมาใส่ จะได้ไม่มี `NaN` เหลือเลย
- `stack(level=1)`:
  - yfinance ส่งข้อมูลมาเป็น MultiIndex (Wide table) กว้าง 102 คอลัมน์ เอาลง relational database ลำบาก
  - ใช้ `stack` ยุบ Ticker จากแนวนอนลงมาเป็นแถวแนวตั้ง (Long format) จะได้ตารางมาตรฐาน 8 คอลัมน์: `date`, `ticker`, `adj_close`, `close`, `high`, `low`, `open`, `volume`

In [4]:
# ดึงข้อมูลรอบสมบูรณ์ (เปลี่ยน SCB.BK เป็น TTB.BK) และ clean format
tickers = [
    "PTT.BK", "PTTEP.BK", "KBANK.BK", "TTB.BK", "BBL.BK", "KTB.BK",
    "CPALL.BK", "BJC.BK", "ADVANC.BK", "AOT.BK", "MINT.BK", "BDMS.BK",
    "BH.BK", "SCC.BK", "CPN.BK", "DELTA.BK", "TDEX.BK"
]

print("กำลังดาวน์โหลดข้อมูลชุดสมบูรณ์...")
raw_data = yf.download(tickers, start="2015-01-01", end="2024-12-31", auto_adjust=False)

# จัดการ missing data และ reshape เป็น long format
clean_data = raw_data.ffill().bfill()
stacked = clean_data.stack(level=1, future_stack=True).reset_index()

# ปรับชื่อคอลัมน์ให้ตรง format SQL
stacked.columns = ['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume']
stacked['date'] = pd.to_datetime(stacked['date']).dt.strftime('%Y-%m-%d')

print("\nขนาดข้อมูลสุทธิที่พร้อมเก็บลงฐานข้อมูล:", stacked.shape)
print("\nตัวอย่างข้อมูล 5 แถวแรก:")
stacked.head()

กำลังดาวน์โหลดข้อมูลชุดสมบูรณ์...

ขนาดข้อมูลสุทธิที่พร้อมเก็บลงฐานข้อมูล: (41310, 8)

ตัวอย่างข้อมูล 5 แถวแรก:


[*********************100%***********************]  17 of 17 completed


,date,ticker,adj_close,close,high,low,open,volume
0,2015-01-05,ADVANC.BK,141.822464,249.000000,254.0,247.000000,252.000000,5232400.0
1,2015-01-05,AOT.BK,24.233814,27.299999,28.0,27.200001,28.000000,24970000.0
2,2015-01-05,BBL.BK,117.724190,190.000000,194.0,189.500000,193.000000,2931800.0
3,2015-01-05,BDMS.BK,13.616664,18.000000,18.0,17.200001,17.299999,41969500.0
4,2015-01-05,BH.BK,105.018890,138.500000,139.0,137.000000,137.000000,569200.0


### Notes: ออกแบบ Database Schema (`data/prices.db`)
- **SQLite**: เป็น local database ไฟล์เดี่ยว ไม่ต้องเปิด server เบาและย้ายง่าย
- **โครงสร้าง 2 ตาราง (เหมือน Excel 1 ไฟล์มี 2 ชีต):**
  1. `tickers`: ตารางข้อมูลหุ้น/Sector (17 แถว)
  2. `prices`: ตารางราคา 8 คอลัมน์ (41,310 แถว)
- **คำศัพท์ SQL ที่ควรรู้:**
  - `stacked.to_sql("prices", ...)`: คำว่า `"prices"` คือ **ชื่อตาราง** ที่เราตั้งขึ้นใน database เพื่อเอา DataFrame ยัดลงไป
  - `cursor = conn.cursor()`: ตัวถือคำสั่ง SQL ไปสั่งรันที่ฐานข้อมูล (เหมือนพนักงานรับออเดอร์)
  - `conn.commit()`: การกดยืนยันเซฟลงดิสก์จริง (เหมือนกด `Cmd + S`) ป้องกันข้อมูลค้างใน RAM
  - `CREATE INDEX ON prices(date, ticker)`: ดัชนีช่วยค้นหา ทำให้ query ข้อมูลตามวัน/หุ้น ไวขึ้นระดับเสี้ยววินาที ไม่ต้องสแกนทั้ง 4 หมื่นแถว

In [5]:
import sqlite3

# 1. สร้าง connection ไปที่ data/prices.db
db_path = "../data/prices.db"
conn = sqlite3.connect(db_path)

# 2. บันทึกตาราง tickers (metadata)
ticker_info = pd.DataFrame([
    {"ticker": "PTT.BK", "name": "PTT Public Co Ltd", "sector": "Energy"},
    {"ticker": "PTTEP.BK", "name": "PTT Exploration & Production", "sector": "Energy"},
    {"ticker": "KBANK.BK", "name": "Kasikornbank", "sector": "Banking"},
    {"ticker": "TTB.BK", "name": "TMBThanachart Bank", "sector": "Banking"},
    {"ticker": "BBL.BK", "name": "Bangkok Bank", "sector": "Banking"},
    {"ticker": "KTB.BK", "name": "Krungthai Bank", "sector": "Banking"},
    {"ticker": "CPALL.BK", "name": "CP ALL", "sector": "Commerce"},
    {"ticker": "BJC.BK", "name": "Berli Jucker", "sector": "Commerce"},
    {"ticker": "ADVANC.BK", "name": "Advanced Info Service", "sector": "ICT"},
    {"ticker": "AOT.BK", "name": "Airports of Thailand", "sector": "Transport"},
    {"ticker": "MINT.BK", "name": "Minor International", "sector": "Tourism"},
    {"ticker": "BDMS.BK", "name": "Bangkok Dusit Medical Services", "sector": "Healthcare"},
    {"ticker": "BH.BK", "name": "Bumrungrad Hospital", "sector": "Healthcare"},
    {"ticker": "SCC.BK", "name": "Siam Cement", "sector": "Construction Materials"},
    {"ticker": "CPN.BK", "name": "Central Pattana", "sector": "Property"},
    {"ticker": "DELTA.BK", "name": "Delta Electronics Thailand", "sector": "Electronic Components"},
    {"ticker": "TDEX.BK", "name": "ThaiDEX SET50 ETF", "sector": "Benchmark"}
])
ticker_info.to_sql("tickers", conn, if_exists="replace", index=False)

# 3. บันทึกตาราง prices (ข้อมูลราคาและ volume)
stacked.to_sql("prices", conn, if_exists="replace", index=False)

# 4. สร้าง Index ให้ค้นหาเร็วขึ้น
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_prices_date_ticker ON prices(date, ticker)")
conn.commit()

print(f"บันทึกข้อมูลลง {db_path} เรียบร้อยแล้ว!")

บันทึกข้อมูลลง ../data/prices.db เรียบร้อยแล้ว!


### Notes: ตรวจสอบข้อมูลด้วย SQL (JOIN & GROUP BY)
- ลองเขียน SQL ดึงข้อมูลจาก 2 ตารางมาประกบกัน เพื่อเช็กว่าแต่ละหุ้นมี 2,430 วันทำการครบไหม

In [6]:
# ทดสอบ Query ข้อมูลผ่าน SQL
query = """
SELECT 
    p.ticker,
    t.sector,
    COUNT(*) as total_days,
    MIN(p.date) as start_date,
    MAX(p.date) as end_date
FROM prices p
JOIN tickers t ON p.ticker = t.ticker
GROUP BY p.ticker
ORDER BY t.sector, p.ticker;
"""

df_check = pd.read_sql(query, conn)
df_check

,ticker,sector,total_days,start_date,end_date
0,BBL.BK,Banking,2430,2015-01-05,2024-12-30
1,KBANK.BK,Banking,2430,2015-01-05,2024-12-30
2,KTB.BK,Banking,2430,2015-01-05,2024-12-30
3,TTB.BK,Banking,2430,2015-01-05,2024-12-30
4,TDEX.BK,Benchmark,2430,2015-01-05,2024-12-30
5,BJC.BK,Commerce,2430,2015-01-05,2024-12-30
6,CPALL.BK,Commerce,2430,2015-01-05,2024-12-30
7,SCC.BK,Construction Materials,2430,2015-01-05,2024-12-30
8,DELTA.BK,Electronic Components,2430,2015-01-05,2024-12-30
9,PTT.BK,Energy,2430,2015-01-05,2024-12-30


### Notes: ทำไมต้องสลับไปมาระหว่าง Long-format (ใน SQL) กับ Wide/Price Matrix (ใน pandas)?

**1. ในฐานข้อมูล SQL (Long-format):**
```text
date       | ticker | price
2015-01-05 | PTT    | 30
2015-01-05 | BBL    | 100
2015-01-06 | PTT    | 33
2015-01-06 | BBL    | 105
```
- ข้อดี: ตารางมี 8 คอลัมน์ตายตัวเสมอ จะมีหุ้นกี่ตัวตารางก็ไม่พัง ค้นหาง่ายมาก เช่น `WHERE ticker = 'PTT.BK'`
- **แต่ทำไมคำนวณการเงินจากตารางนี้ตรงๆ ไม่ได้?**
  - ถ้าสั่ง `pct_change()` มันจะเอาแถวล่างลบแถวบน (เอา BBL 100 ไปเทียบกับ PTT 30 กลายเป็นได้ +233% มั่วทันที)
  - และไม่สามารถหา Correlation ข้ามหุ้นได้เพราะราคากองรวมอยู่ในคอลัมน์เดียวกัน

---

**2. พอจะคำนวณใน Python เราจึงต้อง Pivot ออกมาเป็น Price Matrix (Date x Tickers):**
```text
date       | BBL | PTT
2015-01-05 | 100 | 30
2015-01-06 | 105 | 33
```
- สั่ง `prices.pct_change()`: ได้ return แต่ละหุ้นถูกต้อง (BBL +5%, PTT +10%)
- สั่ง `returns.corr()`: หาเมทริกซ์ Correlation ระหว่างหุ้นข้ามคอลัมน์ได้ทันที!

In [24]:
# ทดสอบโหลดข้อมูลผ่านโมดูล src.data
import sys
sys.path.append("..")

from src.data import load_price_matrix, get_tickers_info

# ดึง Price Matrix ออกมาใช้งาน
prices = load_price_matrix()
print("ขนาด Price Matrix (Days, Tickers):", prices.shape)
print(prices.head())
print(prices.pct_change())

ขนาด Price Matrix (Days, Tickers): (2430, 17)
ticker       ADVANC.BK     AOT.BK      BBL.BK    BDMS.BK       BH.BK  \
date                                                                   
2015-01-05  141.822464  24.233814  117.724190  13.616664  105.018890   
2015-01-06  140.683350  24.145048  117.414436  13.465370  105.777153   
2015-01-07  144.100708  25.121504  118.653618  13.541018  105.777153   
2015-01-08  142.392044  25.654114  118.963409  14.070553  106.535408   
2015-01-09  141.252884  25.831652  117.724190  13.843608  105.018890   

ticker         BJC.BK   CPALL.BK     CPN.BK  DELTA.BK    KBANK.BK     KTB.BK  \
date                                                                           
2015-01-05  29.082920  34.182598  35.039536  5.396035  142.203079  12.433333   
2015-01-06  28.550150  33.981529  35.039536  5.239628  138.324875  12.039516   
2015-01-07  29.263126  34.785820  35.236385  5.787051  140.910355  12.545852   
2015-01-08  29.615690  34.785820  35.630089  5.78